# COMP0178 - Lesson 1: Overview of Databases

- Explained notes covering **all 34 PDF pages** of *1A. Overview of Databases*, Week 1.
- Source: [Lecture_1a.pdf](Lectures/Lecture_1a.pdf), the local lecture file found after the supplied Downloads filename was moved or renamed.
- References below use **PDF page order**, not printed slide numbers: the printed numbering has gaps later in the deck.
- The notes expand the original starter notes on databases, DBMSs, database types, and the SQL coursework.
- Supplementary examples illustrate the lecture concepts; they are not extra coursework requirements.

## Runnable practice: how to use this notebook

- Read a topic, attempt the exercise, run your answer, then open its solution to compare reasoning.
- Choose a **Python 3 Jupyter kernel** and run the setup cell below once per kernel session.
- `%%sql` cells run actual SQL against a disposable **in-memory SQLite** database. No package installation or database server is required beyond the Jupyter Python environment.
- This notebook registers its own small SQL magic; it does not require JupySQL. The SQL practice uses syntax shared by SQLite and MySQL where possible.
- MySQL-only commands such as CREATE DATABASE, USE, ENGINE, GRANT, ANY/ALL, and some ALTER forms remain labelled lecture examples; do not run those against SQLite.
- Each SQL cell runs **one statement**. Python cells run setup/checks; Markdown cells hold notes, questions, and solutions.
- Run an answer cell immediately before its check. Comments-only starter cells prompt you to write a query without revealing the answer.
- `lab.reset()` restores the original dataset; the database is not saved when the kernel closes. Each notebook should run its own setup.
- Checks compare results on this dataset, including duplicates. They do not prove correctness for every possible input.
- To practice MySQL specifically later, use a separate MySQL/JupySQL connection rather than mixing it with this SQLite session.

In [2]:
from pathlib import Path
import sys

# Find the helper whether Jupyter starts in COMP0178 or on the Desktop.
practice_folder = next((p for p in [Path.cwd(), Path.cwd() / 'COMP0178',
    Path('/Users/peidi/Desktop/COMP0178')] if (p / 'sql_practice.py').exists()), None)
if practice_folder is None:
    raise FileNotFoundError('Keep sql_practice.py beside these notebooks')
sys.path.insert(0, str(practice_folder))
from sql_practice import Practice
lab = Practice()
lab.register(get_ipython())

Fresh practice database: Branch 3, Staff 5, Client 3, Property 4, Viewing 4 rows.
%%sql ready. One SQL statement per code cell. Run lab.reset() to restore the sample data.


## 1. What this lecture teaches (pages 1-2, 15-16, 22-23)

- Understand **what databases are** and why organizations use them.
- Distinguish a database from a **database management system (DBMS)**.
- Recognize the main functions and trade-offs of DBMS software.
- Learn the basic table structure of a **relational database**.
- Understand how users communicate with a DBMS and how its components work together.
- Recognize different levels of database description and different client-server roles.
- Understand the module outline and early coursework milestones.

## 2. Why organizations need databases (pages 3-6)

- Organizations produce data about employees, documents, products, sales, and other activities.
- Web applications commonly store users, relationships, posts, photos, account settings, and usage statistics.
- Storing data is only part of the task: applications must also **load, search, combine, update, and analyze it**.
- In a **file-based system**, each application defines and manages its own files and formats.
- Applications may not know about one another's data, making integration difficult.
- Separate files can duplicate information, waste storage, and disagree after updates.
- Different file formats make shared analysis harder.
- The slide-5 example has separate Sales and Contracts applications, each with its own PropertyForRent and Client files.
- Explanatory example: changing a client's address in the Sales file but not the Contracts file leaves inconsistent copies.
- In the slide-6 database approach, both applications access shared PropertyForRent, PrivateOwner, Client, and Lease data **through a DBMS**.
- Central management standardizes data description and can reduce unnecessary duplication and inconsistency.
- Qualification: sharing a database does not automatically eliminate every duplicate or design error; the schema and update rules still matter.

## 3. Database, schema, and DBMS are different things (pages 7-8)

- A **database** is a shared collection of logically related data, together with its description, designed to meet an organization's information needs.
- **Logically related** means the records connect meaningfully: staff work at branches; clients sign leases on properties.
- **Shared** means multiple authorized users or applications can use the same managed data.
- A database is **self-describing**: it includes metadata explaining how its data is organized.
- A **schema** describes that organization: for example, table names, attributes, data types, relationships, and constraints.
- Explanatory distinction: the schema says that a Branch has a branchNo and city; the stored data says that branch B005 is in London.
- A **DBMS** is the software used to define, create, maintain, and control access to the database.
- Example: an auction site's users, auctions, and bids form its database; MySQL or MariaDB can be its DBMS; its table definitions form its schema.
- Changing a row changes the data. Adding a column or defining a new table changes the schema.

### Practice A1

- In your own words, distinguish database, DBMS, schema, and stored records. Use the auction project as an example.
- Write your answer in the Markdown cell below before opening the explanation.

**My answer:**

- A **database** is a collection of logically related data together with its description. In the auction project, it contains information about users, auctions, and bids.
- A **DBMS** is software used to define, manage, and control access to a database, such as MySQL or SQLite. **SQL is the language used to communicate with a DBMS**, not a DBMS itself.
- A **schema** defines the structure and rules of the database: its tables, columns, data types, primary keys, foreign keys, and constraints. For example, a Bid table could have bid_id, user_id, auction_id, and amount columns, with foreign keys linking bids to users and auctions.
- **Stored records** are the actual rows containing data in those tables. For example, one Bid record could say that user 7 placed a £25 bid on auction 12. Adding or changing that record changes the contents, while adding a new column changes the schema.


<details>
<summary>Answer guidance</summary>

- The database contains related users/auctions/bids and their description; MySQL is DBMS software; the schema defines tables/keys; a particular bid is a record.

</details>

## 4. What a DBMS provides (page 9)

- **Data access:** shared methods for retrieving and modifying stored information.
- **Performance optimization:** methods to carry out queries efficiently; an index is a supplementary example.
- **Security:** control who can access data and which operations they may perform.
- **Data integrity:** enforce rules that keep data valid and relationships consistent.
- Explanatory example: reject a Staff record referring to a nonexistent Branch when a suitable foreign-key constraint is declared.
- **Update-failure recovery:** help recover from interrupted or unsuccessful updates.
- Explanatory example: a transaction can avoid leaving a related group of updates only partly completed; later lectures explain transactions and recovery.
- Shared services prevent every application from having to reinvent its own access, security, and recovery mechanisms.
- The trade-offs listed in the lecture are increased **size, complexity, and cost**.
- The usefulness of a DBMS depends on the application's needs, not merely on the amount of data.

## 5. Database types and software products (pages 10-12)

- Different database models organize and handle data differently, with different strengths and limitations.
- **Relational databases** organize data in tables and are the main focus of COMP0178.
- The lecture describes relational DBMSs as mature and dominant; treat this as the lecture's characterization rather than a current market measurement.
- **Hierarchical databases** are introduced as an earlier model; explanatory intuition: records follow parent-child structures.
- **Document-oriented databases** are presented as an alternative, often using JSON-style documents.
- Supplementary illustration: a document can contain a user's profile and nested preferences rather than placing every value in a rectangular row.
- Separate **database model** from **DBMS product**: relational is a model; PostgreSQL is a software implementation of that model.
- Relational product examples in the slides include Oracle, Microsoft Access/SQL Server, IBM Db2, SAP products, MySQL, MariaDB, PostgreSQL, and SQLite.
- **MongoDB is document-oriented**, although it appears beside open-source product names on page 11; do not classify every name on that slide as relational.
- Software may be free, open-source, paid, or proprietary. Cost and licensing are different questions from the data model.
- Page 11's market illustration is credited to William Blair, *Database Software Market: The Long-Awaited Shake-up*, March 2019; it is historical context.

## 6. Module plan and early coursework (pages 13-14)

- **Weeks 1-2:** using relational databases through SQL and learning database design.
- **Weeks 3-6:** relational algebra, storage, query processing/optimization, transaction management, and recovery.
- **Weeks 7-8:** distributed databases.
- **Weeks 9-10:** topics beyond relational DBMSs.
- Coursework contributes **50% of the module mark** and involves designing and using your own SQL database.
- Essential starting material is concentrated in the first two weeks.
- **Week 1:** introduce yourself/form groups and work through SQL exercises on Moodle.
- **Week 2:** learn ER diagrams and normalization; finalize groups and peer-marking criteria by the end of the week.
- **Week 3:** submit the project's ER diagram by the end of the week.
- These are the lecture's week-based milestones, not calendar dates verified against Moodle.
- For detailed auction requirements and submissions, see [coursework_brief.ipynb](coursework_brief.ipynb).

## 7. Relational tables: rows, columns, and domains (pages 17-20)

- Initially, think of a relational database as a **set of two-dimensional tables**.
- A table can represent an entity, such as Branch or Staff, or a relationship.
- A **column** is an attribute/field: for example, branchNo, street, city, or postcode.
- A **row** is a record describing one instance, such as one branch.
- A **domain** is the set of permitted values for an attribute.
- Explanatory example: a salary domain might permit non-negative numeric amounts; a date attribute should contain valid dates.
- A domain is not simply the values currently present: another valid value may be inserted later.
- The slides use capitalized table names and lowerCamelCase attributes as a naming convention, not a universal requirement.
- Their table notation is `Branch(branchNo, street, city, postcode)`.
- **Data definition** specifies the attributes and their domains; **data manipulation** populates and changes records.
- The rectangular layout resembles an Excel table, making it easy to inspect and analyze.
- A relational database also supplies declared relationships, constraints, querying, and managed access; visual similarity to a spreadsheet is only an introduction.

| branchNo | street | city | postcode |
| --- | --- | --- | --- |
| B005 | 22 Deer Rd | London | SW1 4EH |
| B007 | 16 Argyll St | Aberdeen | AB2 3SU |
| B003 | 163 Main St | Glasgow | G11 9QX |
| B004 | 32 Manse Rd | Bristol | BS99 1NZ |
| B002 | 56 Clover Dr | London | NW10 6EU |

- These are the five Branch records shown in the lecture diagram.

## 8. Degree, cardinality, primary keys, and foreign keys (pages 17-19)

- **Degree** of a table = number of columns.
- **Cardinality** of a table = number of rows, excluding its header.
- The pictured Branch table has **degree 4** and **cardinality 5**.
- The pictured Staff table has **degree 8** and **cardinality 6**.
- Adding a record changes cardinality; adding an attribute changes degree.
- A **primary key** is the chosen attribute or combination of attributes identifying each record uniquely; the slides underline it.
- Branch uses **branchNo**. City is unsuitable because both B005 and B002 are in London.
- Supplement: a declared SQL primary key is unique and non-null; a multi-attribute primary key is called composite.
- Page 18 teaches that each table has a primary key as the design convention; SQL software can allow a table without a declared one, so this is not an automatic software guarantee.
- A **foreign key** represents a reference to another table's key.
- In the diagrams, **Staff.branchNo** refers to **Branch.branchNo**, while Staff's own primary key is **staffNo**.
- Multiple staff can share the same branchNo, so a foreign-key value need not be unique in the referencing table.
- Explanatory example: staff SL21 and SL41 both reference B005; their branch address is stored in Branch rather than repeated in every Staff record.
- Qualification: declared foreign keys enforce valid references, normally permitting NULL only when the column's rules allow it.

### Practice A2

- Before running the SQL below, predict Branch’s degree and cardinality, Staff’s degree and cardinality, and whether Staff.branchNo must be unique.
- Write your answer in the Markdown cell below before opening the explanation.

**My answer:**

- **Branch:** degree 3 (3 columns) and cardinality 3 (3 rows).
- **Staff:** degree 6 (6 columns) and cardinality 5 (5 rows).
- **Staff.branchNo does not have to be unique**, because several staff members can work at the same branch. It is a foreign key referencing Branch.branchNo.
- **Branch.branchNo must be unique**, because it is the primary key identifying each branch.
- Degree counts columns; cardinality counts data rows, excluding the header. These counts refer to the notebook's initial practice dataset, which differs from the lecture-slide tables.


In [3]:
%%sql
SELECT * FROM Branch;

branchNo,street,city
B001,1 Main St,London
B002,2 High St,Glasgow
B003,3 Park St,Bristol


3 row(s)


In [4]:
%%sql
SELECT * FROM Staff;

staffNo,fName,lName,position,salary,branchNo
S001,Ada,Lee,Manager,30000,B001
S002,Ben,Patel,Assistant,18000,B001
S003,Cara,Jones,Manager,24000,B002
S004,Dan,Wu,Assistant,NULL,B002
S005,Eve,Smith,Supervisor,20000,B002


5 row(s)


<details>
<summary>Answer guidance</summary>

- Branch: degree 3, cardinality 3. Staff: degree 6, cardinality 5. Staff.branchNo is not unique: several staff can reference one branch.

</details>

## 9. SQL, DDL, and DML (pages 10, 21, 24)

- Relational databases have a formal foundation in **relational algebra/calculus**; these describe operations and queries mathematically.
- **SQL** means Structured Query Language and is standardized, although products can have dialect differences.
- **DDL - Data Definition Language:** define/change database structure, domains, and integrity constraints.
- **DML - Data Manipulation Language:** retrieve, insert, update, and delete stored data.
- SQL provides both kinds of operation. Some classifications call retrieval a separate Data Query Language (DQL); that terminology does not change what SELECT does.
- The slide groups permissions/security with definition facilities; some classifications call permission commands Data Control Language (DCL).
- Supplementary SQL examples below preview the next lecture; they are explanatory, not code extracted from this PDF.

```sql
-- DDL: define the table and primary key.
CREATE TABLE Branch (
    branchNo VARCHAR(4) PRIMARY KEY,
    street VARCHAR(100),
    city VARCHAR(50),
    postcode VARCHAR(10)
);

-- DML: insert a record.
INSERT INTO Branch (branchNo, street, city, postcode)
VALUES ('B005', '22 Deer Rd', 'London', 'SW1 4EH');

-- Retrieval: ask for matching records.
SELECT branchNo, city
FROM Branch
WHERE city = 'London';

-- DML: modify stored data.
UPDATE Branch SET street = '24 Deer Rd' WHERE branchNo = 'B005';
```

- CREATE TABLE changes the schema; INSERT and UPDATE change its contents; SELECT reads the data.
- These fenced snippets are study examples, not automatically executed notebook cells.

## 10. Conceptual, logical, and physical descriptions (page 25)

- A **conceptual schema** describes the organization's data independently of the chosen database model.
- Explanatory example: identify Staff, Branch, and the relationship "staff work at branches" in an ER diagram.
- A **logical schema** expresses the design in a specific model, such as relational tables, attributes, keys, and other metadata.
- Explanatory example: `Staff(staffNo, ..., branchNo)` and `Branch(branchNo, ...)`, with a foreign-key relationship.
- The **physical description** concerns how the data is stored on a particular system: files, pages, indexes, and storage structures.
- These descriptions answer different questions: **what data exists; how the chosen model represents it; how the implementation stores it**.
- The word schema describes structure, not just a diagram or the records themselves.

## 11. Three-schema database architecture and abstraction (pages 26-27)

- The slide calls this "three-tier DB architecture"; the levels are **external, conceptual, and internal**.
- **External level:** user-specific views/subschemas exposing only the relevant or permitted data.
- Explanatory example: a staff directory shows names and branches while excluding salaries.
- **Conceptual level:** the integrated description of the whole database's logical structure, without platform-specific physical details.
- **Internal/physical level:** storage structures and algorithms that organize data on disk.
- Note the terminology: page 25's conceptual-versus-logical distinction concerns design descriptions; the architecture's middle "conceptual" level represents the integrated logical view of the whole database.
- **Abstraction** hides unnecessary implementation details behind an interface.
- **Reduced coupling** means changing one level should require fewer changes elsewhere.
- Supplementary example of physical data independence: adding an index can improve query performance without changing application queries or table meanings.
- Separate applications can have specialized views while sharing integrated data management.

```text
User view 1     User view 2     User view 3
       \           |           /
        Integrated conceptual schema
                     |
              Internal schema
                     |
              Physical storage
```

- Views are abstractions; actual access restrictions also require appropriate permissions.

### Practice A3

- Classify these: a user-specific staff directory; the complete table/relationship description; an index on disk. Then contrast them with browser, PHP application, and DBMS roles.
- Write your answer in the Markdown cell below before opening the explanation.

**My answer:**

- A **user-specific staff directory** is at the **external level**, because it shows a particular user's view of the data.
- The **complete table and relationship description** is at the **conceptual level**, because it defines the structure of the whole database, including tables, relationships, and constraints.
- An **index on disk** is at the **internal level**, because it concerns physical storage and efficient access to records.

For the application roles:

- The **browser** is the **presentation/client tier**: it displays information and receives user input.
- The **PHP application** is the **application tier**: it processes requests and checks business rules.
- The **DBMS** belongs to the **data tier**: it manages data storage and retrieval and enforces database constraints.

The external/conceptual/internal model describes **levels of database description**, while the presentation/application/data model describes **application responsibilities**. They are separate classifications.


<details>
<summary>Answer guidance</summary>

- External view; conceptual/logical integrated schema; internal/physical storage. Browser/application/data server is a separate classification of application responsibilities.

</details>

## 12. Common internal DBMS components (page 28)

- **Query processor:** translates a submitted query into lower-level instructions.
- **Database manager:** coordinates services such as authorization, integrity checks, query optimization, scheduling, temporary storage, and recovery.
- **File manager:** manages the physical data and auxiliary structures, interacting with the operating system.
- These are conceptual roles; real products may divide them into more specialized modules.
- Explanatory sequence for a query: parse/interpret the request, check permissions and plan execution, access storage, then return results.
- Supplementary meaning of optimization: choose an efficient execution plan while preserving the requested result.
- Supplementary meaning of recovery: restore a consistent state after a failed operation or system interruption.

## 13. Multiple users and client-server architecture (pages 29-30)

- A shared database must support requests from multiple users and applications.
- A **client/frontend** sends requests and handles presentation and interaction, possibly with some light processing.
- A **server/backend** handles database queries and storage services.
- In the **three-tier client-server model**, the server responsibilities are split into an application tier and a data tier.
- **Thin client:** displays the interface, such as a web browser.
- **Application server:** handles requests and application/business rules.
- **Data server:** manages storage and retrieval through the DBMS.
- For a PHP/MySQL auction system, a browser is the client, PHP application code handles bid requests, and MySQL/MariaDB manages auction and bid data.
- Explanatory bid flow: browser submits an amount → application checks the request and auction rules → database transaction records a valid bid → result returns to browser.
- The tiers describe responsibilities; they do not require three separate physical computers during development.
- High-traffic data services can further separate a **transaction manager** (schedule updates and handle rollbacks) from a **resource manager** (read/write storage).

```text
Browser / UI <-> Application server <-> Database server
```

- **Do not confuse this model with the external/conceptual/internal model.** Client/application/data concerns application responsibilities; external/conceptual/internal concerns levels of database description.

## 14. Cloud computing and database as a service (pages 31-33)

- The lecture defines cloud computing as on-demand network access to a shared pool of configurable computing resources that can be provisioned and released with little management effort.
- **On-demand/self-service** means users request resources rather than managing all underlying hardware themselves.
- With **in-house management**, an organization runs storage and DBMS software on systems it manages.
- With **Database as a Service (DBaaS)**, a provider hosts/manages database infrastructure while the customer remotely configures and uses the database.
- The slide presents DBaaS as a form of hosted software service; the key lesson is shifting infrastructure responsibility to the provider.
- The customer's schema, queries, data, and appropriate access configuration still require attention; hosting does not design the database for you.
- Provider examples listed in the lecture include AWS, Microsoft Azure, Google Cloud, and cloud offerings from IBM/SAP.
- Providers can offer both relational and non-relational options.
- These are lecture examples, not a comparison of current products or prices.
- Supplementary contrast: local MariaDB for coursework runs on your machine; a managed cloud database is accessed remotely and its physical infrastructure is handled by the provider.

### Practice A4

- Give one reason a managed cloud database still needs careful database design, and one responsibility the provider takes over.
- Write your answer in the Markdown cell below before opening the explanation.

**My answer:**

- Replace this bullet with your explanation.

<details>
<summary>Answer guidance</summary>

- You still choose tables, keys, constraints, and queries. A provider manages underlying hosted infrastructure according to the service; cloud hosting does not fix schema errors.

</details>

## 15. What to remember (page 34)

- **Database = related stored data plus its description.**
- **DBMS = software managing that database and controlled access.**
- **Schema = description of structure; records = current contents.**
- Relational databases organize data in tables and have a mathematical foundation and SQL interface.
- Know **attribute, record, domain, degree, cardinality, primary key, and foreign key**.
- Distinguish **DDL** from **DML**.
- Distinguish **external/conceptual/internal** from **client/application/data** architecture.
- Explain why shared management reduces file-based duplication and supplies common security, integrity, and recovery services.
- The next topic is **SQL**, the standard language for defining and using relational databases.

## 16. Check your understanding

- **1. Is MySQL a database or a DBMS?** Explain using an auction example.
- **2. A table has 4 columns and 20 rows. What are its degree and cardinality?**
- **3. Why is city not a suitable primary key for the pictured Branch table?**
- **4. Why can several Staff rows contain branchNo B005?**
- **5. Which changes the schema: adding a branch record or adding a new attribute?**
- **6. Name one DDL and one DML operation.**
- **7. Which architecture uses external, conceptual, and internal levels?**
- **8. Which tier validates an auction request, and which tier stores the data?**
- **9. Explain one inconsistency caused by separate application files.**
- **10. Does cloud hosting remove the need to design a schema?**

### Answers

- **1.** MySQL is DBMS software; the application's stored users, bids, and auctions form its database.
- **2.** Degree 4; cardinality 20.
- **3.** Multiple branches have city London, so it does not identify a unique row.
- **4.** Staff.branchNo is a foreign key referencing Branch, not a unique identifier for each staff member.
- **5.** Adding an attribute changes the schema; adding a row changes the contents/cardinality.
- **6.** CREATE TABLE is DDL; INSERT or UPDATE is DML.
- **7.** The three-schema database architecture.
- **8.** The application tier handles business rules; the data tier manages storage and database operations.
- **9.** One application updates a client's address while another retains the old address.
- **10.** No. Hosting manages infrastructure; the application still needs an appropriate data model and schema.

## Past-paper connection: foundations for later topics

- See [exam_guide.ipynb](exam_guide.ipynb) for the reviewed 2023-2025 question map and study approach.
- **2023 Q1, 2024 Q1, 2025 Q1:** the schema, keys, and relationships introduced here support later normalization and ER tasks. Know how data requirements lead to entities, relationships, and multiplicities.
- **All Q2:** query processing builds on the DBMS/query-processor overview, then requires SQL, relational algebra, alternative trees, and stated cost assumptions.
- **All Q3:** integrity, multi-user access, and recovery lead to transaction reasoning, not just terminology.
- **All Q4/Q5:** database architecture and model comparisons prepare for distributed and semistructured topics; these later topics require their own lecture notes.
- Practice: explain database versus DBMS, schema versus contents, primary versus foreign key, and external/conceptual/internal versus client/application/data in precise short answers.

## Restore the lab

- Run the Python cell below whenever you want the original sample records back.
- Reset also removes any views or tables you created in this in-memory lab.

In [5]:
lab.reset()

Fresh practice database: Branch 3, Staff 5, Client 3, Property 4, Viewing 4 rows.
